# Part 5 · Notebook 07 — Actions and entry conditions

**Sessions:** S12 (Useful actions & entry conditions) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. Write a crossover with an explicit tie rule and NaN handling.
2. Count bars since an event.
3. Build `within(n)`, and compose conditions like a strategy spec.
4. See how many signals each extra condition removes.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()

In [ ]:
df = p.synthetic_ohlcv(1500, seed=21)
o, h, l, c, v = p.arrays(df)

## 1. Crossover, precisely

"Fast crosses above slow" needs a rule for ties and gaps. Ours: at bar `t`, `a[t] > b[t]` **and** `a[t−1] <= b[t−1]`. So a touch followed by a cross counts once, at the cross. Any NaN among the four values means no signal, and bar 0 is never a cross.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def crossover(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    out = np.zeros(a.shape, dtype=bool)
    with np.errstate(invalid="ignore"):           # comparisons with NaN are False, which is what we want
        out[1:] = ...                             # ✍️ above now, at or below on the previous bar
    return out

cases = [([1, 2, 2, 3, 1], [2, 2, 2, 2, 2]),              # touch, then cross at bar 3
         ([1, 3, 1, 3, 1], [2, 2, 2, 2, 2]),              # two crosses
         ([np.nan, 3, 1, 3, 3], [2, 2, np.nan, 2, 2]),    # NaNs block the signal
         (list(p.ema(c, 10)), list(p.ema(c, 30)))]        # the real thing
mine = [p.attempt(crossover, a, b) for a, b in cases]
mine = p.check("crossover", mine, [p.crossover(a, b) for a, b in cases])
[m[:5].tolist() for m in mine[:3]] + [f"EMA 10 × 30: {int(mine[3].sum())} crosses"]

## 2. Bars since

`bars_since(cond)` is 0 on a bar where `cond` is True, 1 on the next bar, and so on; NaN before the first True. It turns "the golden cross happened recently" into a number you can threshold.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def bars_since(cond):
    out = np.full(len(cond), np.nan)
    last = None
    for i, v in enumerate(cond):
        ...                                       # ✍️ remember the bar of the last True; once there is one, out[i] = i − that bar
    return out

sample = [False, True, False, False, True, False, False, False]
cross = p.crossover(p.ema(c, 10), p.ema(c, 30))
mine = [bars_since(sample), bars_since(cross)]
mine = p.check("bars_since", mine, [p.bars_since(sample), p.bars_since(cross)])
mine[0]

## 3. Conditions that remember: `within(n)`

Entry rules rarely need two events on the *same* bar: "RSI crossed above 30 **within the last 3 bars** and price is above its 200-bar average". `within(cond, n)` is True at `t` if `cond` was True on any of the bars `t−n+1 … t`. A rolling maximum of the booleans does it: `pd.Series(cond).rolling(n, min_periods=1).max()`, back to a boolean array.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def within(cond, n):
    return ...                                    # ✍️

mine = p.attempt(within, sample, 2)
mine = p.check("within", mine, p.within(sample, 2))
mine

## 4. Composing an entry rule

`p.Condition` wraps a boolean array and supports `&`, `|`, `~`, `.within(n)` and `.confirm(n)` (True for `n` bars in a row). A strategy's entry rule then reads like its spec. Watch how each condition thins the signals.

In [ ]:
r = p.rsi(c, 14)
rsi_up = p.Condition(p.crossover(r, np.full_like(r, 30.0)), "rsi_x_30")
uptrend = p.Condition(c > p.sma(c, 200), "above_sma200")
narrow = p.Condition(p.nr_n(h, l, 7), "nr7")
inside = p.Condition(p.inside_bar(h, l), "inside")

steps = [rsi_up, rsi_up.within(3), rsi_up.within(3) & uptrend.confirm(5), rsi_up.within(3) & uptrend.confirm(5) & (narrow | inside)]
for s in steps:
    print(s)
entry = steps[2].values
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(c, lw=0.8, color="#8a8984", label="close"); ax.plot(p.sma(c, 200), label="SMA 200")
ax.plot(np.flatnonzero(entry), c[entry], "^", color=p.PALETTE[2], label="entry bars")
ax.set_title("RSI crossed above 30 within 3 bars, in a confirmed uptrend"); ax.legend(); plt.show()

Note that `within(3)` turns one event into up to three True bars. Before backtesting, decide whether you want an **entry event** (use `crossover` on the combined condition) or a **state** you hold while it is True.

## Wrap-up

* Every action states its tie and NaN rules; test them with tiny hand-made arrays.
* `bars_since`, `within`, `confirm` turn events into states you can combine.
* Graded version: `labs/part05/week19_patterns` (crossovers, gaps, inside bars, NR-n, new highs, `bars_since`, the `Condition` class).